# vLLM — High-Throughput LLM Inference with PagedAttention

---

## What Is This Notebook About?

**vLLM** is a fast, memory-efficient inference engine for large language models. When you want to serve an open-source LLM (Llama, Mistral, Gemma, etc.) to many users simultaneously, vLLM is the industry-standard solution. It achieves 2-24× higher throughput than naive HuggingFace inference, making it possible to serve LLMs cost-effectively.

By the end of this notebook you will understand:
- Why LLM inference is memory-constrained and what the KV cache is
- PagedAttention — the key innovation in vLLM
- Continuous batching — why it beats static batching
- How to serve LLMs with vLLM's OpenAI-compatible API
- Quantization for running large models on small GPUs
- A mini-project: building a load-balanced LLM serving setup

---

## Real-World Analogy: Restaurant Kitchen

Imagine a restaurant kitchen serving 100 diners:

**Without vLLM (naive batching)**: The chef waits for an entire table of 8 to order, prepares all 8 dishes together, serves the table, then takes the next table's order. If table 3 has a quick order (salad) and table 7 has a complex one (steak), the fast table waits for the slow one.

**With vLLM (continuous batching)**: The kitchen runs continuously. As soon as one dish is served, the next order starts immediately. Fast orders come out quickly; slow orders don't block others. The kitchen is always at full capacity.

**PagedAttention**: Instead of reserving a huge prep counter for each dish "just in case", the kitchen efficiently shares counter space across all dishes currently being prepared, only using what's needed at each moment.

---

## Prerequisites
- Basic Python
- OpenAI SDK notebook (API calls pattern)
- Understanding that LLMs need GPUs with VRAM

---

## Table of Contents
1. Installation & Setup
2. The LLM Memory Problem
3. PagedAttention — vLLM's Key Innovation
4. Continuous Batching
5. Offline Inference
6. OpenAI-Compatible Server
7. Quantization — Running Large Models on Small GPUs
8. Performance Benchmarks
9. vLLM vs Alternatives
10. Common Pitfalls
11. Mini Project: LLM Serving Pipeline Design
12. Interview Q&A
13. Resources

---

## Official Resources
- **Docs**: https://docs.vllm.ai/
- **GitHub**: https://github.com/vllm-project/vllm
- **PagedAttention Paper**: https://arxiv.org/abs/2309.06180
- **YouTube (vLLM talk)**: https://www.youtube.com/watch?v=80bIUggRJf4
- **Supported Models**: https://docs.vllm.ai/en/latest/models/supported_models.html

## 1. Installation & Setup

In [ ]:
# Install (requires NVIDIA GPU with CUDA):
# pip install vllm

# For CPU-only (very slow, only for testing):
# pip install vllm --extra-index-url https://download.pytorch.org/whl/cpu

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

try:
    from vllm import LLM, SamplingParams
    VLLM_AVAILABLE = True
    print("vLLM available!")
except ImportError:
    VLLM_AVAILABLE = False
    print("vLLM not installed (requires NVIDIA GPU + CUDA).")
    print("Install: pip install vllm")
    print("All cells simulate output for learning purposes.")

# Check GPU availability
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    if GPU_AVAILABLE:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("No GPU detected (CPU only — vLLM needs GPU for production)")
except ImportError:
    GPU_AVAILABLE = False
    print("PyTorch not installed")

CAN_RUN = VLLM_AVAILABLE and GPU_AVAILABLE
print(f"\nvLLM: {'✓' if VLLM_AVAILABLE else '✗'}  GPU: {'✓' if GPU_AVAILABLE else '✗'}")
print("Note: All concepts are explained with code + simulated output.")
print("Setup complete!")

## 2. The LLM Memory Problem

### Why is LLM inference hard?

When an LLM generates text, it uses the **KV cache** (Key-Value cache) — a memory buffer that stores intermediate computations for all tokens seen so far. This is what allows autoregressive generation without recomputing everything from scratch each token.

**The problem**: The KV cache grows with sequence length and is allocated PER REQUEST.

```
KV cache memory = batch_size × seq_len × n_layers × d_head × 2 × dtype_bytes

For Llama-2-7B:
  n_layers = 32, d_head = 128, dtype = float16 (2 bytes)
  For seq_len=2048: 2048 × 32 × 128 × 2 × 2 = ~33.5 MB PER REQUEST

If you reserve memory for max_seq_len upfront:
  8 concurrent requests × 33.5 MB = 268 MB just for KV caches
  But most requests use much less than the max!
  → 60-80% of reserved memory is WASTED on average
```

This fragmentation and pre-allocation is why naive serving systems are so inefficient.

In [ ]:
# ── KV Cache Memory Analysis ──────────────────────────────────────────

def calculate_kv_cache_size_mb(model_config, seq_len, batch_size=1, dtype_bytes=2):
    """
    Calculate KV cache memory requirement.
    Formula: 2 × n_layers × n_heads × head_dim × seq_len × batch_size × dtype_bytes
    The 2x is for K and V (two matrices).
    """
    n_layers = model_config['n_layers']
    n_heads = model_config['n_heads']
    head_dim = model_config['head_dim']
    # Memory in bytes, convert to MB
    memory_bytes = 2 * n_layers * n_heads * head_dim * seq_len * batch_size * dtype_bytes
    return memory_bytes / (1024 ** 2)

# Model configurations
MODELS = {
    'Llama-2-7B':  {'n_layers': 32,  'n_heads': 32,  'head_dim': 128, 'param_gb': 14.0},
    'Llama-2-13B': {'n_layers': 40,  'n_heads': 40,  'head_dim': 128, 'param_gb': 26.0},
    'Mistral-7B':  {'n_layers': 32,  'n_heads': 8,   'head_dim': 128, 'param_gb': 14.0},
    'Llama-2-70B': {'n_layers': 80,  'n_heads': 64,  'head_dim': 128, 'param_gb': 140.0},
}

print("KV Cache Memory Analysis")
print("=" * 80)
print(f"{'Model':<15} {'Params':>8} {'seq=512':>10} {'seq=2048':>10} {'seq=8192':>10} {'Waste%':>8}")
print("-" * 80)

for model_name, config in MODELS.items():
    kv_512  = calculate_kv_cache_size_mb(config, 512)
    kv_2048 = calculate_kv_cache_size_mb(config, 2048)
    kv_8192 = calculate_kv_cache_size_mb(config, 8192)

    # Average actual usage is ~30% of max
    waste = 70  # 70% waste on average with naive pre-allocation

    print(f"{model_name:<15} {config['param_gb']:>7.0f}GB {kv_512:>9.1f}MB {kv_2048:>9.1f}MB {kv_8192:>9.1f}MB {waste:>7}%")

print()
print("Problem: With naive serving, you MUST pre-allocate memory for max_seq_len")
print("  If max_seq_len=2048, but avg request is 500 tokens: 75% memory wasted")
print("  This limits how many concurrent requests you can serve")
print()

# Visualize: Memory waste in naive vs paged approach
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Naive allocation
ax = axes[0]
request_lengths = [150, 800, 400, 1200, 300, 600, 900, 250]  # Actual token counts
max_len = 2048
allocated = [max_len] * len(request_lengths)  # Each reserves max_len

y = range(len(request_lengths))
ax.barh(y, allocated, color='lightcoral', alpha=0.7, label='Pre-allocated (wasted)')
ax.barh(y, request_lengths, color='steelblue', alpha=0.9, label='Actually used')

total_waste = sum(max_len - r for r in request_lengths) / sum(allocated) * 100
ax.set_title(f'Naive KV Cache Allocation\n{total_waste:.0f}% Memory Wasted!', fontweight='bold')
ax.set_xlabel('KV Cache Tokens')
ax.set_ylabel('Request #')
ax.legend()
ax.axvline(max_len, color='red', linestyle='--', linewidth=1.5, label='max_seq_len')
ax.grid(True, alpha=0.3, axis='x')

# Right: PagedAttention
ax2 = axes[1]
# Show page-based allocation
page_size = 256
pages_needed = [max(1, r // page_size + 1) for r in request_lengths]
pages_allocated = pages_needed  # Only allocate what's needed!

ax2.barh(y, [p * page_size for p in pages_needed], color='steelblue', alpha=0.9, label='Pages allocated')
ax2.barh(y, request_lengths, color='steelblue', alpha=0.5, label='Actual usage', hatch='//')

total_waste_paged = sum(max(0, p * page_size - r) for r, p in zip(request_lengths, pages_needed)) / \
                   sum(p * page_size for p in pages_needed) * 100
ax2.set_title(f'PagedAttention (vLLM)\n~{total_waste_paged:.0f}% Internal Fragmentation (minimal!)', fontweight='bold')
ax2.set_xlabel('KV Cache Tokens')
ax2.set_ylabel('Request #')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='x')

plt.suptitle('Memory Efficiency: Naive vs PagedAttention', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/vllm_memory.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Naive allocation: {total_waste:.0f}% memory wasted per request")
print(f"PagedAttention:   Only ~{total_waste_paged:.0f}% internal fragmentation")
print(f"Improvement:      ~{total_waste/max(total_waste_paged,1):.0f}x more efficient use of GPU VRAM")

## 3. PagedAttention — vLLM's Core Innovation

**PagedAttention** solves the KV cache fragmentation problem by borrowing an idea from operating systems: **virtual memory paging**.

In operating systems:
- Physical memory is divided into fixed-size **pages**
- Programs address **virtual** memory; the OS maps virtual pages to physical pages
- Pages are allocated on-demand, not pre-allocated

In PagedAttention:
- GPU memory is divided into fixed-size **KV blocks** (each block holds KV cache for e.g., 16 tokens)
- Each request has a **block table** that maps its virtual sequence to physical blocks
- Blocks are allocated on-demand as tokens are generated
- **Key innovation**: Different requests can share blocks (e.g., if they share a common prefix like a system prompt)

```
Request A: [Block 0, Block 3, Block 7]        ← Non-contiguous physical blocks
Request B: [Block 0, Block 1, Block 5]        ← Shares Block 0 (same system prompt!)
Request C: [Block 2, Block 4, Block 6, Block 8]

Block table maps: virtual position → physical block
```

**Result**: Nearly zero wasted memory, and prefix caching (shared blocks for common prefixes) for free.

In [ ]:
# ── PagedAttention Visualization ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Colors for different requests
req_colors = {'A': '#3498db', 'B': '#e74c3c', 'C': '#2ecc71', 'free': '#ecf0f1'}
req_border = {'A': '#2980b9', 'B': '#c0392b', 'C': '#27ae60', 'free': '#bdc3c7'}

# Physical GPU memory as a grid of blocks
n_blocks = 20
grid_cols = 5
grid_rows = 4

# PagedAttention block allocation
# Requests share Block 0 (same system prompt = prefix caching)
allocation = {
    0: 'A/B',  # Shared prefix block!
    1: 'A', 2: 'A', 3: 'A',
    4: 'B', 5: 'B',
    6: 'C', 7: 'C', 8: 'C', 9: 'C', 10: 'C',
    11: 'free', 12: 'free', 13: 'free', 14: 'free',
    15: 'free', 16: 'free', 17: 'free', 18: 'free', 19: 'free',
}

ax = axes[0]
ax.set_title('PagedAttention: Physical Memory\n(Non-contiguous blocks per request)', fontweight='bold')
ax.axis('off')

for block_id in range(n_blocks):
    row = block_id // grid_cols
    col = block_id % grid_cols
    owner = allocation[block_id]

    if owner == 'A/B':
        # Shared block - show as gradient
        color = '#9b59b6'
        border = '#6c3483'
    elif owner == 'free':
        color = req_colors['free']
        border = req_border['free']
    else:
        color = req_colors[owner]
        border = req_border[owner]

    rect = mpatches.FancyBboxPatch(
        (col * 1.2, (grid_rows - 1 - row) * 1.2), 1.0, 1.0,
        boxstyle='round,pad=0.05', facecolor=color, edgecolor=border, linewidth=2
    )
    ax.add_patch(rect)
    ax.text(col * 1.2 + 0.5, (grid_rows - 1 - row) * 1.2 + 0.5,
            f'B{block_id}\n{owner}', ha='center', va='center',
            fontsize=8, fontweight='bold', color='white' if owner != 'free' else '#555')

ax.set_xlim(-0.2, grid_cols * 1.2 + 0.2)
ax.set_ylim(-0.2, grid_rows * 1.2 + 0.8)

# Legend
legend_patches = [
    mpatches.Patch(color='#3498db', label='Request A (4 blocks)'),
    mpatches.Patch(color='#e74c3c', label='Request B (3 blocks)'),
    mpatches.Patch(color='#2ecc71', label='Request C (5 blocks)'),
    mpatches.Patch(color='#9b59b6', label='Shared Prefix (A+B)'),
    mpatches.Patch(color='#ecf0f1', label='Free blocks (8 available)'),
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=8)
ax.text(3, -0.15, 'Block 0 shared by A and B (same system prompt = prefix caching!)',
        ha='center', fontsize=8, style='italic', color='#9b59b6')

# Block tables (virtual → physical mapping)
ax2 = axes[1]
ax2.axis('off')
ax2.set_title('Block Tables: Virtual → Physical Mapping', fontweight='bold')

block_tables = [
    ('Request A', ['A/B→B0', 'A→B1', 'A→B2', 'A→B3', '—', '—'], '#3498db'),
    ('Request B', ['A/B→B0', 'B→B4', 'B→B5', '—', '—', '—'], '#e74c3c'),
    ('Request C', ['C→B6', 'C→B7', 'C→B8', 'C→B9', 'C→B10', '—'], '#2ecc71'),
]

for i, (req_name, table, color) in enumerate(block_tables):
    y = 0.75 - i * 0.28
    ax2.text(0.0, y + 0.1, req_name, fontsize=11, fontweight='bold', color=color, transform=ax2.transAxes)
    for j, entry in enumerate(table):
        x = 0.05 + j * 0.15
        bg = color if entry != '—' else '#ecf0f1'
        rect = mpatches.FancyBboxPatch((x, y - 0.03), 0.13, 0.1,
                                        boxstyle='round,pad=0.01',
                                        facecolor=bg, edgecolor='gray', linewidth=1,
                                        transform=ax2.transAxes, clip_on=False)
        ax2.add_patch(rect)
        ax2.text(x + 0.065, y + 0.02, entry, ha='center', va='center', fontsize=7,
                 fontweight='bold', color='white' if entry != '—' else '#aaa',
                 transform=ax2.transAxes)

ax2.text(0.5, 0.08,
         'Blocks need NOT be contiguous in physical memory!\nThe GPU kernel handles the mapping transparently.',
         ha='center', va='center', fontsize=9, style='italic', color='gray', transform=ax2.transAxes,
         bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray'))

plt.suptitle('PagedAttention: Virtual Memory for KV Caches', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/vllm_paged_attention.png', dpi=100, bbox_inches='tight')
plt.show()

print("PagedAttention benefits:")
print("  1. No waste: blocks allocated only when needed (on-demand paging)")
print("  2. No fragmentation: any free block can be assigned to any request")
print("  3. Prefix caching: shared system prompt = shared blocks = free KV cache reuse")
print("  4. Result: 2-24x more concurrent requests on same hardware")

## 4. Continuous Batching vs Static Batching

### Static Batching (naive approach):
```
Batch 1: [req1, req2, req3, req4] ─── generate until all DONE → serve all
                                          Problem: if req1 is done in 10 tokens
                                          but req4 needs 500, req1 WAITS for req4!

Batch 2: [req5, req6, req7, req8] ─── repeat
```

### Continuous Batching (vLLM):
```
Iteration 1: [req1, req2, req3, req4] → generate 1 token each
Iteration 2: [req1, req2, req3, req4] → generate 1 token each
...
Iteration 10: req1 finishes → IMMEDIATELY replace with req5
Iteration 10: [req5, req2, req3, req4] → req5 starts without waiting!
```

The batch is **continuously refilled** as requests complete. GPU utilization approaches 100%.

In [ ]:
# ── Continuous Batching Visualization ────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
fig.suptitle('Static Batching vs Continuous Batching (vLLM)', fontsize=13, fontweight='bold')

# Simulate request lengths (tokens to generate)
np.random.seed(42)
n_requests = 10
request_lengths = np.random.randint(20, 200, n_requests)
request_names = [f'R{i}' for i in range(n_requests)]

colors = plt.cm.Set3(np.linspace(0, 1, n_requests))

# ── Static Batching ───────────────────────────────────────────────────
ax = axes[0]
batch_size = 4

timeline = []  # (request_id, start_time, end_time, batch_num)
current_time = 0
for batch_start in range(0, n_requests, batch_size):
    batch = list(range(batch_start, min(batch_start + batch_size, n_requests)))
    batch_duration = max(request_lengths[b] for b in batch)  # Wait for slowest!
    for req_id in batch:
        timeline.append((req_id, current_time, current_time + batch_duration, current_time + request_lengths[req_id]))
    current_time += batch_duration

for req_id, start, batch_end, actual_end in timeline:
    # Time actually generating
    ax.barh(req_id, actual_end - start, left=start, color=colors[req_id], alpha=0.9, height=0.6)
    # Time wasted waiting
    if batch_end > actual_end:
        ax.barh(req_id, batch_end - actual_end, left=actual_end, color='lightgray', alpha=0.7, height=0.6,
                hatch='///', edgecolor='white')

ax.set_yticks(range(n_requests))
ax.set_yticklabels(request_names)
ax.set_xlabel('Time (tokens)')
ax.set_title(f'Static Batching (batch_size={batch_size}): Total time = {current_time} tokens', fontweight='bold')
legend_elements = [mpatches.Patch(color='steelblue', label='Generating'),
                   mpatches.Patch(color='lightgray', hatch='///', label='Waiting (wasted!)')]
ax.legend(handles=legend_elements, loc='upper right')
ax.grid(True, alpha=0.3, axis='x')

# ── Continuous Batching ───────────────────────────────────────────────
ax2 = axes[1]

# Simulate continuous batching
# All requests start immediately when a slot opens
slots = [0] * batch_size  # Current end time of each slot
cb_timeline = []

for req_id in range(n_requests):
    # Find the slot that finishes earliest
    earliest_slot = np.argmin(slots)
    start = slots[earliest_slot]
    end = start + request_lengths[req_id]
    slots[earliest_slot] = end
    cb_timeline.append((req_id, start, end))

total_cb_time = max(end for _, _, end in cb_timeline)

for req_id, start, end in cb_timeline:
    ax2.barh(req_id, end - start, left=start, color=colors[req_id], alpha=0.9, height=0.6)

ax2.set_yticks(range(n_requests))
ax2.set_yticklabels(request_names)
ax2.set_xlabel('Time (tokens)')
ax2.set_title(f'Continuous Batching (vLLM): Total time = {total_cb_time} tokens '
              f'({(1 - total_cb_time/current_time)*100:.0f}% faster!)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/tmp/vllm_batching.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Static batching:     Total time = {current_time} tokens")
print(f"Continuous batching: Total time = {total_cb_time} tokens")
print(f"Speedup: {current_time/total_cb_time:.1f}x faster with same hardware!")

## 5. Offline Inference — Batch Processing

In [ ]:
# ── vLLM Offline Inference (Batch Mode) ──────────────────────────────

print("=== vLLM Offline Inference ===")
print()
print("Offline inference = process a batch of prompts without a server")
print("Use case: generate summaries for 10,000 documents overnight")
print()

if CAN_RUN:
    # Load model
    llm = LLM(
        model='facebook/opt-125m',   # Tiny model for demo (125M params)
        max_model_len=512,
        gpu_memory_utilization=0.5   # Use 50% of GPU memory
    )

    # Define sampling parameters
    sampling_params = SamplingParams(
        temperature=0.7,   # Randomness
        top_p=0.95,        # Nucleus sampling
        max_tokens=100,    # Max output tokens
        stop=["\n\n"]      # Stop at double newline
    )

    # Process batch of prompts
    prompts = [
        "The capital of France is",
        "Python is a programming language that",
        "The most important invention of the 20th century was",
        "Climate change is caused by",
        "The future of artificial intelligence is",
    ]

    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - t0

    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)

    for output in outputs:
        prompt = output.prompt
        generated = output.outputs[0].text
        print(f"Prompt: {prompt[:50]}")
        print(f"Output: {generated[:100]}")
        print()

    print(f"Batch size: {len(prompts)} prompts")
    print(f"Total generated: {total_tokens} tokens")
    print(f"Time: {elapsed:.2f}s")
    print(f"Throughput: {total_tokens/elapsed:.0f} tokens/second")

else:
    code = '''
from vllm import LLM, SamplingParams

# Load the model (downloads automatically from HuggingFace Hub)
llm = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    max_model_len=4096,
    gpu_memory_utilization=0.85,   # Use 85% of GPU VRAM
    tensor_parallel_size=1,        # How many GPUs to use (1 for single GPU)
)

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=512,
    stop=["</s>", "[INST]"]   # Stop tokens for Llama
)

# Process 5 prompts in one batch
prompts = [
    "Explain quantum computing in simple terms:",
    "Write a haiku about Python:",
    "What is the capital of France?",
    "Summarize the French Revolution in 2 sentences:",
    "What are 3 benefits of exercise?",
]

outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(f"Prompt: {output.prompt}")
    print(f"Output: {output.outputs[0].text}")
    print()
    '''
    print(code)
    print("Simulated output:")
    simulated_outputs = [
        ("Explain quantum computing in simple terms:",
         "Quantum computing uses quantum bits (qubits) that can be 0 and 1 simultaneously, enabling massive parallel processing for specific problems like optimization and cryptography."),
        ("Write a haiku about Python:",
         "Indented loops dance\nPython whispers in the shell\nBug fixed, code runs free"),
        ("What is the capital of France?",
         "Paris is the capital and largest city of France."),
    ]
    for prompt, output in simulated_outputs:
        print(f"Prompt: {prompt}")
        print(f"Output: {output}")
        print()
    print("Performance (A100 GPU, Llama-2-7B):")
    print("  5 prompts, avg 200 tokens each = 1000 total tokens")
    print("  Time: ~3.2 seconds")
    print("  Throughput: ~312 tokens/second")
    print("  vs HuggingFace naive: ~45 tokens/second → 7x faster!")

## 6. OpenAI-Compatible Server

In [ ]:
# ── vLLM OpenAI-Compatible Server ─────────────────────────────────────

print("=== vLLM OpenAI-Compatible API Server ===")
print()
print("vLLM can serve as a drop-in replacement for the OpenAI API!")
print("Any code that uses the OpenAI SDK works with vLLM without changes.")
print()

print("Step 1: Start the server")
print("=" * 50)
print('''
# Terminal command:
python -m vllm.entrypoints.openai.api_server \\
    --model meta-llama/Llama-2-7b-chat-hf \\
    --host 0.0.0.0 \\
    --port 8000 \\
    --max-model-len 4096 \\
    --gpu-memory-utilization 0.85

# Server starts and shows:
# INFO: Started server process
# INFO: Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
''')

print("Step 2: Use exactly like OpenAI SDK (just change base_url!)")
print("=" * 50)
print('''
from openai import OpenAI

# Point to your local vLLM server instead of OpenAI
client = OpenAI(
    api_key="EMPTY",           # No real API key needed
    base_url="http://localhost:8000/v1"
)

# EXACT same code as OpenAI — just different base_url!
response = client.chat.completions.create(
    model="meta-llama/Llama-2-7b-chat-hf",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain RAG in 2 sentences."}
    ],
    temperature=0.7,
    max_tokens=200,
    stream=True  # Streaming also works!
)

for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end='', flush=True)
''')

print()
print("Benefits of OpenAI-compatible API:")
print("  ✓ Zero code changes from OpenAI to vLLM")
print("  ✓ Use open-source models (Llama, Mistral, Gemma) at $0 API cost")
print("  ✓ Data privacy — model runs on YOUR servers")
print("  ✓ Predictable costs — pay for hardware, not per token")
print("  ✓ Customizable — fine-tune the model for your use case")

# Available endpoints
print()
print("Available endpoints (matches OpenAI API):")
endpoints = [
    ('POST', '/v1/chat/completions', 'Chat completion (GPT-style)'),
    ('POST', '/v1/completions', 'Text completion (old-style)'),
    ('GET',  '/v1/models', 'List available models'),
    ('POST', '/v1/embeddings', 'Generate embeddings (if model supports)'),
    ('GET',  '/health', 'Server health check'),
    ('GET',  '/metrics', 'Prometheus metrics (for monitoring)'),
]

for method, path, desc in endpoints:
    print(f"  [{method:4}] {path:<30} {desc}")

## 7. Quantization — Running Large Models on Small GPUs

In [ ]:
# ── Quantization Overview ─────────────────────────────────────────────

print("=" * 65)
print(" Quantization: Fitting Large Models on Consumer GPUs")
print("=" * 65)
print()
print("LLMs store weights as 32-bit floats by default.")
print("Quantization reduces precision → smaller model → fits on smaller GPU")
print()

# Memory requirements for Llama-2-7B
model_gb = 14  # Approximate for Llama-2-7B in fp16

quantization_info = {
    'fp32 (full precision)': {'bits': 32, 'size_gb': model_gb * 2, 'quality': 100, 'speed': 70},
    'fp16 (half precision)': {'bits': 16, 'size_gb': model_gb, 'quality': 99.9, 'speed': 100},
    'int8 (AWQ/GPTQ)':       {'bits': 8,  'size_gb': model_gb / 2, 'quality': 99.5, 'speed': 90},
    'int4 (GPTQ/GGUF)':      {'bits': 4,  'size_gb': model_gb / 4, 'quality': 98.0, 'speed': 80},
    'int3 (experimental)':   {'bits': 3,  'size_gb': model_gb / 5, 'quality': 95.0, 'speed': 70},
}

print(f"{'Method':<25} {'Bits':>5} {'Size (7B)':>12} {'Quality%':>10} {'Speed%':>8} {'Fits on':<20}")
print("-" * 82)

for method, info in quantization_info.items():
    if info['size_gb'] <= 8:
        fits = 'RTX 3070 (8GB)'
    elif info['size_gb'] <= 16:
        fits = 'RTX 3080 (16GB)'
    elif info['size_gb'] <= 24:
        fits = 'RTX 3090 (24GB)'
    elif info['size_gb'] <= 40:
        fits = 'A100 (40GB)'
    else:
        fits = '2× A100 needed'

    print(f"{method:<25} {info['bits']:>5} {info['size_gb']:>11.1f}GB {info['quality']:>10.1f} {info['speed']:>8} {fits:<20}")

print()
print("vLLM quantization options:")
print('''
# AWQ (Activation-aware Weight Quantization) — best quality int4
llm = LLM(
    model="TheBloke/Llama-2-7B-chat-AWQ",
    quantization="awq",         # AWQ
    dtype="auto"
)

# GPTQ — GPU-accelerated int4
llm = LLM(
    model="TheBloke/Llama-2-7B-chat-GPTQ",
    quantization="gptq",
    dtype="auto"
)
''')

# Plot quality vs memory trade-off
fig, ax = plt.subplots(figsize=(10, 5))

methods = list(quantization_info.keys())
sizes = [info['size_gb'] for info in quantization_info.values()]
qualities = [info['quality'] for info in quantization_info.values()]
colors_q = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

scatter = ax.scatter(sizes, qualities, c=colors_q, s=200, zorder=5)
for method, size, quality, color in zip(methods, sizes, qualities, colors_q):
    ax.annotate(method, (size, quality), textcoords='offset points',
                xytext=(5, 5), fontsize=8, color=color)

ax.axvline(8, color='green', linestyle='--', alpha=0.5, label='RTX 3070 (8GB)')
ax.axvline(16, color='blue', linestyle='--', alpha=0.5, label='RTX 3080 (16GB)')
ax.axvline(24, color='purple', linestyle='--', alpha=0.5, label='RTX 3090 (24GB)')

ax.set_title('Quantization: Quality vs Memory Trade-off (Llama-2-7B)', fontweight='bold')
ax.set_xlabel('Model Size (GB VRAM needed)')
ax.set_ylabel('Quality (% of fp16 baseline)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(93, 101)

plt.tight_layout()
plt.savefig('/tmp/vllm_quantization.png', dpi=100, bbox_inches='tight')
plt.show()

print("Recommendation:")
print("  For production: fp16 (best quality, needs larger GPU)")
print("  For consumer GPU (RTX 3070/3080): AWQ int4 (best quality/size ratio)")
print("  For CPU inference (llama.cpp): GGUF format (not vLLM, but the standard for CPU)")

## 8. Common Pitfalls

In [ ]:
print("=" * 68)
print(" vLLM Common Pitfalls")
print("=" * 68)

pitfalls = [
    {
        "title": "1. Not enough GPU VRAM → OOM error",
        "fix": "Reduce gpu_memory_utilization=0.7 or use quantization (AWQ/GPTQ)",
        "why": "vLLM pre-allocates GPU memory. If model + KV cache exceed VRAM → crash."
    },
    {
        "title": "2. max_model_len too large → all KV cache blocks used immediately",
        "fix": "Set max_model_len to actual max context you need (e.g., 2048 not 131072)",
        "why": "KV cache size = max_model_len × batch_size × model_params. Smaller = more concurrent requests."
    },
    {
        "title": "3. Wrong model name/path",
        "fix": "Use exact HuggingFace Hub ID: 'meta-llama/Llama-2-7b-chat-hf'",
        "why": "vLLM auto-downloads from HuggingFace Hub. Model must be public or you need HF_TOKEN."
    },
    {
        "title": "4. Not setting HF_TOKEN for gated models (Llama, Gemma)",
        "fix": "export HF_TOKEN=your_huggingface_token (get from huggingface.co/settings/tokens)",
        "why": "Llama-2, Llama-3, and Gemma require accepting license agreement and HF token."
    },
    {
        "title": "5. Running vLLM on CPU (extremely slow)",
        "fix": "Use llama.cpp or ollama for CPU inference; vLLM is optimized for GPU",
        "why": "vLLM uses CUDA kernels. Without GPU, it falls back to CPU: 100x slower."
    },
    {
        "title": "6. Not using sampling_params.stop tokens → runaway generation",
        "fix": "Always set stop=[<eos_token>, '###', '\\n\\n'] appropriate for your model",
        "why": "Without stop tokens, model generates until max_tokens. Wastes time and GPU."
    },
]

for p in pitfalls:
    print(f"\n{'─'*68}")
    print(f"  {p['title']}")
    print(f"  Fix:  {p['fix']}")
    print(f"  Why:  {p['why']}")

print(f"\n{'='*68}")

## 9. Mini Project: LLM Serving Architecture Design

In [ ]:
# ── Mini Project: Production LLM Serving Design ───────────────────────

# Simulate and compare serving strategies

class LLMServingSimulator:
    """
    Simulates throughput of different LLM serving approaches.
    """

    def __init__(self, gpu_memory_gb=40, model_size_gb=14):
        self.gpu_memory_gb = gpu_memory_gb
        self.model_size_gb = model_size_gb
        self.available_for_kv = gpu_memory_gb - model_size_gb

    def calculate_max_batch_size(self, avg_seq_len, kv_size_per_token_mb=0.033):
        """How many concurrent requests can we handle?"""
        kv_per_request_mb = avg_seq_len * kv_size_per_token_mb
        available_mb = self.available_for_kv * 1024
        return int(available_mb / kv_per_request_mb)

    def estimate_throughput(self, batch_size, tokens_per_step=1, steps_per_second=50):
        """Tokens/second throughput estimate."""
        return batch_size * tokens_per_step * steps_per_second


sim = LLMServingSimulator(gpu_memory_gb=40, model_size_gb=14)

# Compare scenarios
scenarios = [
    ('Short responses (100 tokens avg)', 100),
    ('Medium responses (500 tokens avg)', 500),
    ('Long responses (2000 tokens avg)', 2000),
    ('RAG responses (4000 tokens avg)', 4000),
]

print("=" * 70)
print(" LLM Serving Capacity Analysis (A100 40GB, Llama-2-7B)")
print("=" * 70)
print(f"{'Scenario':<35} {'Max Batch':>10} {'Throughput':>12} {'Requests/min':>14}")
print("-" * 70)

results = []
for scenario, avg_len in scenarios:
    max_batch = sim.calculate_max_batch_size(avg_len)
    throughput = sim.estimate_throughput(max_batch)
    req_per_min = throughput * 60 / avg_len
    results.append((scenario, avg_len, max_batch, throughput, req_per_min))
    print(f"{scenario:<35} {max_batch:>10} {throughput:>11}/s {req_per_min:>14.0f}")

print()
print("Key insight: Short outputs = higher batch size = higher throughput")
print("For chatbots: avg response ~200 tokens → can serve ~100+ concurrent users")
print()

# Production architecture recommendations
print("=" * 70)
print(" Production vLLM Deployment Architecture")
print("=" * 70)
print('''
Load Balancer (nginx/caddy)
    │
    ├── vLLM Server 1 (GPU 1, A100 40GB)
    │     └── LLM Model: Llama-2-7B-chat
    │
    ├── vLLM Server 2 (GPU 2, A100 40GB)
    │     └── LLM Model: Llama-2-7B-chat
    │
    └── vLLM Server 3 (GPU 3, A100 40GB)
          └── LLM Model: Llama-2-7B-chat

Total throughput: 3× single server
High availability: if one server fails, others handle the load
Cost: ~$10/hr for 3× A100 on AWS/GCP vs ~$0.002/1K tokens for GPT-3.5
Break-even: if you generate >5M tokens/day, self-hosting is cheaper
''')

# Visualize throughput comparison
labels = [s.split('(')[0].strip() for s, _, _, _, _ in results]
req_per_min = [r for _, _, _, _, r in results]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, req_per_min, color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'], alpha=0.85)

for bar, val in zip(bars, req_per_min):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:.0f}/min', ha='center', fontsize=11, fontweight='bold')

ax.set_title('vLLM: Request Throughput by Response Length\n(A100 40GB, Llama-2-7B)', fontweight='bold')
ax.set_ylabel('Requests per Minute')
ax.set_xlabel('Response Length Category')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig('/tmp/vllm_throughput.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Interview Q&A

---

### Q1: What is PagedAttention and why is it important?
**A**: PagedAttention is vLLM's core innovation, borrowed from OS virtual memory management. The KV cache (Key-Value cache that stores intermediate computations during generation) is divided into fixed-size blocks (pages) instead of being pre-allocated as contiguous memory. These blocks are stored in a block table that maps virtual positions to physical GPU memory locations. Benefits: (1) No memory waste from pre-allocation, (2) No fragmentation since any free block serves any request, (3) Prefix sharing — requests with the same system prompt can share KV cache blocks. Result: 2-24× higher throughput vs naive implementation.

---

### Q2: What is the KV cache and why does it matter for inference?
**A**: KV cache stores the Key and Value attention tensors computed for all previous tokens during autoregressive generation. Without it, generating each new token would require recomputing attention over the entire sequence from scratch — O(n²) per token. With KV cache, we compute Q (query) for the new token and use cached K,V from all previous tokens — O(n) per token. The downside: KV cache grows with sequence length and consumes GPU VRAM, limiting how many concurrent requests a server can handle. This is exactly the problem PagedAttention solves.

---

### Q3: What is continuous batching and why is it better than static batching?
**A**: Static batching: collect a fixed batch of N requests, process them all until every request finishes, then start the next batch. The problem: slow requests block fast ones; the GPU waits for the slowest request. Continuous batching: the batch is dynamically updated — when any request finishes, a new request immediately takes its slot. The GPU is always at full capacity. vLLM implements this at the iteration level: after each token generation step, finished sequences are replaced with new requests from the queue. This achieves near 100% GPU utilization vs ~30-60% with static batching.

---

### Q4: What is quantization in the context of LLMs?
**A**: Quantization reduces the numerical precision of model weights to save memory. Full precision (fp32) uses 4 bytes per weight. fp16 uses 2 bytes (same quality, 2× smaller). int8 uses 1 byte (~1% quality loss, 2× smaller than fp16). int4 uses 0.5 bytes (~2-5% quality loss, 4× smaller than fp16). For a 7B parameter model: fp16 = 14GB VRAM, int4 = 3.5GB. int4 makes it possible to run Llama-2-7B on a consumer RTX 3070 (8GB). Best methods: AWQ (Activation-aware Weight Quantization) and GPTQ for minimal quality loss.

---

### Q5: When would you use vLLM vs calling the OpenAI API?
**A**: Use OpenAI API when: startup speed matters (no GPU needed), volume is low, you need the latest models (GPT-4o), or simplicity is paramount. Use vLLM when: (1) Data privacy is critical — data cannot leave your servers, (2) High volume — at >5M tokens/day, self-hosting is cheaper, (3) Customization — you need a fine-tuned model, (4) Latency — on-premises avoids network round-trip, (5) Open-source preference — Llama, Mistral, Gemma are free to use. vLLM enables serving open-source models with production-grade performance.

---

### Q6: What is tensor parallelism and when do you need it?
**A**: Tensor parallelism splits individual layers of the model across multiple GPUs — each GPU holds part of each weight matrix. Useful when the model doesn't fit on a single GPU (e.g., 70B models need 4× A100s). In vLLM: `tensor_parallel_size=4` splits across 4 GPUs. Contrast with pipeline parallelism (different layers on different GPUs — has inter-layer communication overhead) and data parallelism (same model on multiple GPUs, different batches — no sharing needed). For most use cases: use the largest single GPU you can afford; tensor parallelism only when model doesn't fit on one.

## 11. Resources

### Official
- **vLLM Docs**: https://docs.vllm.ai/
- **GitHub**: https://github.com/vllm-project/vllm
- **Supported Models**: https://docs.vllm.ai/en/latest/models/supported_models.html

### Tutorials
- **vLLM Talk (Woosuk Kwon)**: https://www.youtube.com/watch?v=80bIUggRJf4
- **Serving Llama with vLLM**: https://www.youtube.com/watch?v=rl9a2BPBPu4

### Papers
- **PagedAttention (vLLM paper)**: https://arxiv.org/abs/2309.06180
- **FlashAttention-2**: https://arxiv.org/abs/2307.08691
- **AWQ (quantization)**: https://arxiv.org/abs/2306.00978

### Alternatives
- **TGI (Text Generation Inference)**: https://github.com/huggingface/text-generation-inference
- **llama.cpp (CPU)**: https://github.com/ggerganov/llama.cpp
- **Ollama (local)**: https://ollama.ai/

---

## Summary

| Concept | Takeaway |
|---------|----------|
| KV Cache | Stores attention K,V tensors; grows with sequence length |
| PagedAttention | Non-contiguous KV blocks, on-demand allocation; ~0% waste |
| Continuous batching | Refill batch on completion; near 100% GPU utilization |
| Offline inference | `LLM.generate(prompts, sampling_params)` for batch jobs |
| API server | OpenAI-compatible; drop-in replacement with `base_url` |
| Quantization | int4 (AWQ/GPTQ) = 4× smaller model, ~2% quality loss |
| vs OpenAI | vLLM = privacy + cost at scale; OpenAI = simplicity |

**Next**: DeepSpeed — train and fine-tune massive LLMs efficiently with ZeRO optimization!